## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 5b: Cum decid modelele de clasificare: kNN, arbori, ansambluri, SVM</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/05-clasificare/curs-05b-clasificare.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "05-clasificare"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Cum lucrăm azi: tu, agentul și notebook-ul

Acest curs se joacă în trei: **lecția** (acest notebook), **tu** și **agentul tău** (Gemini în Colab, un agent din VS Code sau din Antigravity). Agentul scrie codul; tu decizi ce se cere, de ce, și verifici dacă rezultatul are sens. Fiecare idee este un **capitol** cu aceiași pași:

| Pas | Ce faci |
|---|---|
| **Concept** | citești 2-3 propoziții |
| **Prezice** | spui (sau scrii) ce crezi că va ieși, *înainte* să rulezi |
| **Cere agentului** | formulezi cererea; apoi rulezi codul de referință și îl compari cu ce a scris agentul |
| **Verifică** | compari cu valorile de control |
| **Capcană** | vezi ce poate merge prost, chiar dacă codul rulează |
| **Pentru client** | spui rezultatul în limbaj de manager |

**Primer** (lipește-l agentului la începutul conversației; dacă ai pierdut conversația sau agentul a rămas fără tokeni, lipește primerul și *Jurnalul de sesiune* de la finalul notebook-ului unui alt agent și scrie „continuă de aici”):

```text
Rolul tău: ești partenerul meu de lucru la cursul SEED1 (analiză de date și învățare automată), modulul 5: clasificare (regresie logistică, prag de decizie, matricea de confuzie, kNN, arbori de decizie, random forest, gradient boosting, SVM). Eu sunt student și nu știu să programez în Python; eu decid ce se face și de ce, tu scrii codul, iar eu verific rezultatul.

Datele cursului:
- Se încarcă DOAR cu `from seed.date import incarca`. `incarca("lab-set1")[0]` întoarce tabelul clienților unui operator de telecomunicații (Telco Customer Churn); ținta este coloana `Churn` (Yes / No).
- Nu folosi căi de fișiere și nu descărca date de pe internet.
- Folosește pandas, matplotlib/seaborn și scikit-learn. Cod scurt, comentarii în română, un pas pe rând.

Reguli pentru acest modul (le aplici chiar dacă nu le cer eu):
1. Scoate ID-urile (`customerID`). Verifică tipurile: coloanele numerice stocate ca text (de exemplu `TotalCharges`) se convertesc, iar valorile goale se tratează explicit, cu o explicație.
2. Împarte train/test stratificat după țintă (20% test, `random_state=42`) ÎNAINTE de orice transformare învățată din date; pregătirea și modelul stau într-un `Pipeline`.
3. Compară întotdeauna cu modelul de referință (clasa majoritară). Raportează acuratețea pe antrenare și pe test, plus recall, precizie și AUC pe test pentru clasa „pleacă”; nu te opri la acuratețe.
4. Standardizează datele pentru regresia logistică, kNN și SVM; arborii și ansamblurile de arbori nu au nevoie.
5. Când compari modele, folosește aceeași împărțire și aceleași metrici pentru toate, într-un singur tabel.
6. Nu inventa coloane și spune-mi dacă ceva pare ciudat (acuratețe aproape perfectă pe antrenare, un model sub cel de referință), în loc să-l prezinți ca pe un succes.

Cum lucrăm:
- Implicit ești în MOD TUTORE: înainte de a-mi da codul, îmi pui o singură întrebare scurtă despre concept („ce crezi că se va întâmpla?”). Nu rezolvi temele în locul meu; mă ajuți să verific.
- La cererea mea poți trece în MOD EXECUTANT (execuți direct și spui pe scurt ce ai făcut) sau în MOD RECENZENT (cauți greșeli în ce îți arăt: ID-uri printre predictori, evaluare pe datele de antrenare, transformări învățate înainte de împărțire).
- Când scriu „închide capitolul”, completezi Jurnalul de sesiune în formatul de mai jos, numai din ce am discutat. Dacă ceva nu s-a discutat, scrii „nu știu”.

Formatul Jurnalului de sesiune:
## Jurnal de sesiune
- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):

Răspunde acum doar cu: rolul tău în două propoziții și cele 6 reguli, pe scurt.
```

## Unde am rămas

Proiectul vostru cere **cel puțin trei modele** de clasificare. Ca să le alegeți și să le explicați, trebuie să știți **cum decide fiecare**: ce „vede”, cum poate greși și ce parametru îl face mai simplu sau mai complicat. Pentru fiecare model:
- desenăm **frontiera de decizie** pe doar două coloane (vechimea și factura lunară), ca s-o putem vedea: roșu = „pleacă”, albastru = „rămâne”;
- îl evaluăm apoi cu **toate** coloanele, prin aceeași funcție `evalueaza`, care adaugă un rând în **clasament** (ca în `curs-04b`).

Azi la curs înțelegem modelele; antrenarea pe setul vostru de date o faceți la seminar, în aceeași arenă.

**Planul de azi (≈55 min):** capitolul 5 vecinii (kNN) · 6 arborele de decizie · 7 ansamblurile (random forest, gradient boosting) · 8 SVM · 9 arena · ce îi spunem clientului.

Celula de mai jos refă pașii din `curs-05a` și definește recuzita: `pregatire`, `evalueaza` și `frontiera`.

In [ ]:
# Unde am rămas în curs-05a: date, împărțire, recuzită (rulați această celulă)
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from seed.date import incarca
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

date = incarca("lab-set1")[0]
df = date.drop(columns=["customerID"])                         # ID: nu spune nimic despre un client nou
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)   # 11 clienți noi: n-au plătit încă
y = (df.pop("Churn") == "Yes").astype(int)                     # ținta: 1 = pleacă, 0 = rămâne
X = df
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def pregatire(scalare=False):
    """Rețeta de pregătire, învățată DOAR pe antrenare (în Pipeline).
    Numere: mediana în locul valorilor lipsă. Text: „Lipsă” în locul valorilor lipsă, apoi one-hot
    (categoriile noi din test sunt ignorate). La cerere: standardizarea tuturor coloanelor rezultate."""
    text = make_pipeline(SimpleImputer(strategy="constant", fill_value="Lipsă"),
                         OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    coloane = make_column_transformer((SimpleImputer(strategy="median"), make_column_selector(dtype_include="number")),
                                      (text, make_column_selector(dtype_exclude="number")),
                                      verbose_feature_names_out=False)
    return make_pipeline(coloane, StandardScaler()) if scalare else coloane

clasament = []

def evalueaza(model, nume):
    """Antrenează modelul pe aceeași antrenare și adaugă în clasament metricile pe același test."""
    start = time.time()
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    scor = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    clasament.append({"model": nume,
                      "acuratețe antrenare": round(model.score(X_train, y_train), 3),
                      "acuratețe test": round(accuracy_score(y_test, p), 3),
                      "recall test": round(recall_score(y_test, p), 3),
                      "precizie test": round(precision_score(y_test, p, zero_division=0), 3),
                      "AUC test": round(roc_auc_score(y_test, scor), 3),
                      "secunde": round(time.time() - start, 1)})
    return model

DOUA = ["tenure", "MonthlyCharges"]

def frontiera(model, titlu, ax):
    """Antrenează modelul DOAR pe vechime și factura lunară și colorează harta: roșu = «pleacă», albastru = «rămâne»."""
    model.fit(X_train[DOUA], y_train)
    xx, yy = np.meshgrid(np.linspace(0, 72, 120), np.linspace(18, 120, 120))
    grila = pd.DataFrame({"tenure": xx.ravel(), "MonthlyCharges": yy.ravel()})
    z = model.predict_proba(grila)[:, 1] if hasattr(model, "predict_proba") else model.predict(grila)
    ax.contourf(xx, yy, z.reshape(xx.shape), levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.7)
    e = X_test.sample(300, random_state=0)
    ax.scatter(e["tenure"], e["MonthlyCharges"], c=y_test[e.index], cmap="RdBu_r", s=10, edgecolors="k", linewidths=0.3)
    ax.set_title(f"{titlu}\nacuratețe test (2 coloane): {model.score(X_test[DOUA], y_test):.3f}", fontsize=10)
    ax.set_xlabel("vechime (luni)")
    ax.set_ylabel("factura lunară")

evalueaza(DummyClassifier(strategy="most_frequent"), "referință: clasa majoritară")
evalueaza(make_pipeline(pregatire(scalare=True), LogisticRegression(max_iter=1000)), "regresie logistică")
fig, ax = plt.subplots(figsize=(5, 4))
frontiera(make_pipeline(StandardScaler(), LogisticRegression()), "regresie logistică", ax)
plt.tight_layout()
plt.show()
pd.DataFrame(clasament)

**Verifică.** Referința: acuratețe 0,735, recall 0, AUC 0,5. Regresia logistică: acuratețe 0,807, recall 0,567, AUC 0,842 (ca în `curs-05a`). Pe hartă, frontiera regresiei logistice este o **linie dreaptă**: clienții noi cu factură mare sunt în zona roșie.

## Capitolul 5. Vecinii (kNN)

**Concept.** kNN (*k nearest neighbors*) nu învață o formulă: **memorează** clienții din antrenare. Pentru un client nou caută cei $k$ clienți cei mai apropiați (distanța din `curs-03a`, capitolul 1) și ia votul lor; proporția de „pleacă” printre vecini este probabilitatea. $k$ mic: frontieră zimțată, care urmărește fiecare client (memorează). $k$ mare: frontieră netedă, care poate pierde detalii. Pentru că se bazează pe distanță, kNN cere **standardizare**.

**Prezice.** Ce acuratețe pe antrenare are kNN cu $k = 1$? Ce se întâmplă cu kNN pe toate coloanele dacă uităm standardizarea?

**Cere agentului:**

> Desenează frontiera kNN pe `tenure` și `MonthlyCharges` (standardizate) pentru k = 1 și k = 25. Apoi adaugă în clasament kNN cu k = 25 pe toate coloanele, o dată fără standardizare și o dată cu standardizare.

In [ ]:
# Cod de referință: kNN
from sklearn.neighbors import KNeighborsClassifier

fig, axe = plt.subplots(1, 2, figsize=(10, 4))
frontiera(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)), "kNN, k = 1", axe[0])
frontiera(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)), "kNN, k = 25", axe[1])
plt.tight_layout()
plt.show()

evalueaza(make_pipeline(pregatire(), KNeighborsClassifier(n_neighbors=25)), "kNN k=25, fără scalare")
evalueaza(make_pipeline(pregatire(scalare=True), KNeighborsClassifier(n_neighbors=25)), "kNN k=25, cu scalare")
pd.DataFrame(clasament).tail(2)

**Verifică.** Pe două coloane: k = 1 dă o hartă „pătată”, cu acuratețe 0,977 pe antrenare (a memorat) și 0,717 pe test; k = 25 o hartă netedă, 0,786 pe test. Pe toate coloanele: fără scalare acuratețe 0,790, dar recall 0,385 și AUC 0,800; cu scalare acuratețe 0,776, recall 0,567, AUC 0,812.

**Capcană.** Fără standardizare, distanța e dominată de `TotalCharges` (valori de la 0 la ~8.700): ceilalți 18 predictori aproape nu contează. Codul rulează fără nicio eroare, iar **acuratețea iese chiar mai mare** fără scalare; abia recall-ul (0,39 față de 0,57) arată că modelul prinde mult mai puțini clienți care pleacă. kNN mai are o problemă la seturi mari (proiectele cu sute de mii de rânduri): fiecare predicție caută prin toată antrenarea, deci e lent.

**Pentru client.** „kNN spune: «clienții care seamănă cu acesta au plecat în proporție de X%»; ușor de explicat, dar depinde mult de cum măsurăm asemănarea.”

## Capitolul 6. Arborele de decizie

**Concept.** Arborele pune **întrebări da/nu** (de exemplu „contract lunar?”, „vechime sub 6 luni?”) și împarte clienții în grupuri tot mai „pure”. La fiecare pas alege întrebarea care separă cel mai bine plecările de rămâneri (Referința R2). Un arbore mic se citește ca un set de **reguli**, ușor de explicat unui manager. Fără limită, pune întrebări până izolează fiecare client: **memorează**. Parametrul `max_depth` (adâncimea maximă) îl ține simplu.

**Prezice.** Care va fi prima întrebare a arborelui? Ce acuratețe pe antrenare are un arbore fără limită de adâncime? Dar pe test?

**Cere agentului:**

> Antrenează un arbore de decizie cu `max_depth=3` pe toate coloanele (în `Pipeline`, fără scalare) și desenează-l cu numele coloanelor. Apoi desenează frontiera pe cele două coloane pentru adâncimea 3 și fără limită și adaugă în clasament arborele fără limită și pe cel cu `max_depth=5`.

In [ ]:
# Cod de referință: un arbore mic, citit ca reguli
from sklearn.tree import DecisionTreeClassifier, plot_tree

arbore3 = make_pipeline(pregatire(), DecisionTreeClassifier(max_depth=3, random_state=42)).fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(15, 5.5))
plot_tree(arbore3[-1], feature_names=arbore3[0].get_feature_names_out(), class_names=["rămâne", "pleacă"],
          filled=True, impurity=False, proportion=True, fontsize=7, ax=ax)
plt.show()

In [ ]:
# Cod de referință: adâncimea limitată vs. nelimitată
fig, axe = plt.subplots(1, 2, figsize=(10, 4))
frontiera(DecisionTreeClassifier(max_depth=3, random_state=42), "arbore, adâncime 3", axe[0])
frontiera(DecisionTreeClassifier(random_state=42), "arbore fără limită", axe[1])
plt.tight_layout()
plt.show()

evalueaza(make_pipeline(pregatire(), DecisionTreeClassifier(random_state=42)), "arbore fără limită")
evalueaza(make_pipeline(pregatire(), DecisionTreeClassifier(max_depth=5, random_state=42)), "arbore, max_depth=5")
pd.DataFrame(clasament).tail(2)

**Verifică.** Prima întrebare: `Contract_Month-to-month` (contract lunar?). Ramura cea mai riscantă: contract lunar, internet pe fibră și vechime de cel mult 13 luni. Arborele fără limită: acuratețe 0,998 pe antrenare și 0,720 pe test, **sub referința** de 0,735 (AUC 0,647). Cu `max_depth=5`: 0,804 și 0,798, AUC 0,830.

**Capcană.** Arborele nelimitat a memorat aproape fiecare client din antrenare și pe clienți noi e mai prost decât a spune „nimeni nu pleacă”. Pe hartă se vede: frontiera e fărâmițată în zeci de dreptunghiuri mici. Un agent care lasă parametrii impliciți la `DecisionTreeClassifier()` exact asta produce. Arborii nu au nevoie de scalare: o întrebare „vechime < 6?” nu depinde de unități.

**Pentru client.** „Un arbore mic dă reguli clare (de exemplu «contract lunar, internet pe fibră și client de cel mult un an → risc mare»), pe care echipa de retenție le poate folosi direct.”

## Capitolul 7. Ansamblurile: mulți arbori

**Concept.** Un singur arbore e instabil: schimbi câțiva clienți și se schimbă toate întrebările. Soluția e să folosim **mulți arbori**:
- **Random forest** (pădure aleatoare): sute de arbori adânci, fiecare antrenat pe un eșantion diferit de clienți și, la fiecare întrebare, pe o submulțime aleatoare de coloane; **votează**. Greșelile lor individuale se compensează.
- **Gradient boosting**: arbori mici, construiți **unul după altul**; fiecare corectează greșelile celor de dinainte.

Sunt de obicei cei mai preciși pe date tabelare, dar nu mai pot fi citiți ca reguli. Putem afla în schimb ce coloane au contat cel mai mult (**importanța**).

**Prezice.** Random forest folosește arbori fără limită de adâncime. Va memora și el?

**Cere agentului:**

> Desenează frontierele pentru random forest și gradient boosting pe cele două coloane. Adaugă în clasament random forest (300 de arbori) și `HistGradientBoostingClassifier` pe toate coloanele (`random_state=42`) și afișează cele mai importante 8 coloane după random forest.

In [ ]:
# Cod de referință: random forest și gradient boosting
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

fig, axe = plt.subplots(1, 2, figsize=(10, 4))
frontiera(RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1), "random forest", axe[0])
frontiera(HistGradientBoostingClassifier(random_state=42), "gradient boosting", axe[1])
plt.tight_layout()
plt.show()

padure = evalueaza(make_pipeline(pregatire(), RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)), "random forest")
evalueaza(make_pipeline(pregatire(), HistGradientBoostingClassifier(random_state=42)), "gradient boosting")
print(pd.DataFrame(clasament).tail(2).to_string(index=False))

importanta = pd.Series(padure[-1].feature_importances_, index=padure[0].get_feature_names_out())
print("\nCele mai importante coloane (random forest):")
importanta.sort_values(ascending=False).head(8).round(3)

**Verifică.** Random forest: acuratețe 0,998 pe antrenare și 0,784 pe test, AUC 0,820. Gradient boosting: 0,878 / 0,791, AUC 0,833. Cele mai importante coloane: `TotalCharges`, `tenure`, `MonthlyCharges`, apoi contractul lunar.

**Capcană.** Random forest are acuratețe aproape perfectă pe antrenare (fiecare arbore e adânc), dar pe test se ține bine: votul a sute de arbori diferiți nu memorează ca un singur arbore. Deci un scor mare pe antrenare **nu e singur o dovadă** de supraînvățare; contează testul. Și: „importanța” arată ce coloane folosește modelul, nu ce *cauzează* plecarea.

**Pentru client.** „Modelele cu mulți arbori sunt printre cele mai precise; ne spun ce contează (vechimea, factura, tipul de contract), dar nu ca reguli simple.”

## Capitolul 8. SVM: marginea cea mai largă

**Concept.** SVM (*support vector machine*) caută frontiera care lasă **cea mai largă bandă** (margine) între clase. Doar clienții de lângă margine (**vectorii suport**) o determină. Cu un **kernel** (de obicei `rbf`), frontiera se poate curba. Parametrul `C` controlează cât de strictă e marginea: `C` mare = încearcă să clasifice corect fiecare client din antrenare (risc de memorare); `C` mic = margine largă, mai tolerantă. Ca kNN, SVM cere **standardizare**. Pe seturi foarte mari (sute de mii de rânduri) devine lent.

**Prezice.** Cum va arăta frontiera SVM cu kernel `rbf` față de cea a regresiei logistice?

**Cere agentului:**

> Desenează frontiera unui SVM cu kernel `rbf` (standardizat) pe cele două coloane, pentru `C=0.1` și `C=10`. Adaugă în clasament un SVM `rbf` pe toate coloanele, cu standardizare.

In [ ]:
# Cod de referință: SVM
from sklearn.svm import SVC

fig, axe = plt.subplots(1, 2, figsize=(10, 4))
frontiera(make_pipeline(StandardScaler(), SVC(kernel="rbf", C=0.1)), "SVM rbf, C = 0,1", axe[0])
frontiera(make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10)), "SVM rbf, C = 10", axe[1])
plt.tight_layout()
plt.show()

evalueaza(make_pipeline(pregatire(scalare=True), SVC(kernel="rbf")), "SVM (rbf)")
pd.DataFrame(clasament).tail(1)

**Verifică.** Pe două coloane, frontiera SVM e curbă (nu dreaptă, ca la logistică); C = 10 o face mai sinuoasă. Pe toate coloanele: acuratețe test 0,787, recall 0,497, AUC 0,795.

**Capcană.** SVM nu dă direct probabilități (harta e în două culori, nu în nuanțe), iar AUC se calculează din distanța până la frontieră. Dacă agentul cere `probability=True`, antrenarea devine de câteva ori mai lentă. Detalii, cu exemple vizuale: `extra-05-svm`.

**Pentru client.** „SVM e un model precis pe date de mărime medie, dar greu de explicat; îl alegem doar dacă bate clar modelele mai simple.”

## Capitolul 9. Arena

**Concept.** Toate modelele au trecut prin aceeași funcție `evalueaza`: **aceeași împărțire, aceeași pregătire, aceleași metrici**. Doar așa comparația e cinstită. Schimbarea modelului a fost de fiecare dată **o singură linie**. Exact asta faceți în proiect: cel puțin trei modele, în același tabel, cu referința pe primul rând.

**Prezice.** Care model câștigă la AUC? Este același cu cel care câștigă la recall?

**Cere agentului:**

> Afișează clasamentul ordonat după AUC pe test și un grafic cu AUC și recall pentru fiecare model. Care model ai recomanda și de ce, ținând cont și de cât de ușor e de explicat?

In [ ]:
# Cod de referință: clasamentul final
tabel = pd.DataFrame(clasament).sort_values("AUC test", ascending=False).reset_index(drop=True)
display(tabel)

ax = tabel.set_index("model")[["AUC test", "recall test"]].iloc[::-1].plot.barh(figsize=(7, 4.5))
ax.set_xlim(0, 1)
ax.set_title("Arena: același test, aceleași metrici")
plt.tight_layout()
plt.show()

**Verifică.** Primele locuri la AUC: regresia logistică (0,842), gradient boosting (0,833), arborele cu `max_depth=5` (0,830), random forest (0,820). Arborele fără limită (0,647) și referința (0,5) sunt la coada clasamentului.

**Capcană.** **Cel mai simplu model a câștigat.** Pe acest set, relațiile sunt aproape liniare, iar modelele mai complicate nu au ce să adauge; pe alt set (poate pe al vostru) ordinea poate fi alta, de aceea proiectul cere cel puțin trei modele. Diferențele de câteva miimi de AUC între primele modele pot ține de împărțire (`curs-04b`, capitolul 5): nu declarați un câștigător pe o diferență de 0,005. Iar cu pragul 0,5, toate modelele au recall modest; pragul se reglează după costuri (`curs-05a`, capitolul 4; modulul 6). La egalitate, câștigă **modelul mai ușor de explicat**.

**Pentru client.** Vezi secțiunea următoare.

## Ce îi spunem clientului

**Cere agentului:** „Scrie 3 propoziții pentru managerul de retenție: ce model recomanzi și de ce, cât de bine prinde clienții care pleacă și ce factori contează.”

In [ ]:
# Rezumat construit din rezultatele de mai sus
modele = tabel[~tabel["model"].str.startswith("referință")]
primul = modele.iloc[0]
logist = tabel.set_index("model").loc["regresie logistică"]
if primul["model"] == "regresie logistică":
    print(f"1. Am comparat {len(modele)} modele pe aceiași {len(y_test)} clienți de test; cel mai bun la AUC este chiar cel mai simplu, "
          f"regresia logistică ({logist['AUC test']:.3f}): îl recomandăm, fiind și cel mai ușor de explicat.")
else:
    print(f"1. Am comparat {len(modele)} modele pe aceiași {len(y_test)} clienți de test; cel mai bun la AUC este {primul['model']} ({primul['AUC test']:.3f}), "
          f"iar regresia logistică ({logist['AUC test']:.3f}) e mai ușor de explicat.")
print(f"2. La pragul 0,5, regresia logistică prinde {logist['recall test']:.0%} din clienții care pleacă; cu un prag mai mic prinde mai mulți, cu mai multe oferte trimise degeaba.")
print(f"3. Contează cel mai mult: {', '.join(importanta.sort_values(ascending=False).head(3).index)} (după random forest).")

---
# Referință

Definiții și detalii pentru acasă. Nu se parcurg la curs.

## R1. Modelele pe scurt

| Model | Cum decide | Parametrul principal | Scalare? | Explicabil? | Atenție la |
|---|---|---|---|---|---|
| Regresie logistică | combinație liniară + sigmoidă | `C` (regularizare) | da | da (coeficienți) | frontiere doar liniare |
| kNN | votul celor $k$ vecini | `n_neighbors` | **da** | parțial („seamănă cu…”) | lent pe date mari; coloane irelevante |
| Arbore de decizie | întrebări da/nu | `max_depth`, `min_samples_leaf` | nu | **da** (reguli) | memorează fără limită |
| Random forest | votul a sute de arbori | `n_estimators`, `max_features` | nu | importanța coloanelor | mai lent, fișiere mari |
| Gradient boosting | arbori mici, corectându-se unul pe altul | `learning_rate`, `max_iter` | nu | importanța coloanelor | supraînvață dacă rulează prea mult |
| SVM | marginea cea mai largă, cu kernel | `C`, `gamma` | **da** | nu | lent pe date mari; fără probabilități directe |

Alegerea hiperparametrilor (de exemplu cel mai bun `max_depth`) se face prin validare încrucișată: modulul 6.

## R2. Cum alege arborele întrebarea

La fiecare nod, arborele încearcă toate coloanele și pragurile și o păstrează pe cea care face grupurile rezultate cât mai **pure**. Puritatea se măsoară cu **indicele Gini**, $G = 1 - \sum_k p_k^2$ (0 = toți din aceeași clasă), sau cu **entropia**, $H = -\sum_k p_k \log_2 p_k$ (`criterion="gini"` sau `"entropy"`). Exemplu: un grup cu 50% plecări are $G = 0{,}5$; unul cu 10% plecări are $G = 0{,}18$.

## R3. Bagging și boosting

- **Bagging** (random forest): modele independente, antrenate pe eșantioane cu revenire (*bootstrap*) din antrenare; se face media sau votul. Scade **varianța** (instabilitatea) arborilor adânci.
- **Boosting** (gradient boosting, XGBoost, LightGBM): modele mici, antrenate secvențial, fiecare pe erorile celor anterioare. Scade **bias-ul** (prea multă simplitate) al arborilor mici.

## R4. Kernelul SVM, intuitiv

Un kernel calculează asemănarea dintre doi clienți ca și cum datele ar fi fost transformate într-un spațiu cu mai multe dimensiuni, în care o frontieră dreaptă devine posibilă, fără să facă efectiv transformarea. `rbf` (gaussian) consideră doi clienți asemănători dacă sunt apropiați; `gamma` spune cât de repede scade asemănarea cu distanța.

## R5. Pentru aprofundare

- [extra-05-knn-arbori-pima.ipynb](extra-05-knn-arbori-pima.ipynb): kNN și arbori pe datele Pima (diabet): alegerea lui $k$, vizualizarea și „toaletarea” arborelui, random forest. Materialul din anii anteriori.
- [extra-05-svm.ipynb](extra-05-svm.ipynb) și slide-urile [extra-05-svm.pdf](extra-05-svm.pdf): separabilitate, vectori suport, efectul lui `C` și al kernelului, SVM pentru regresie.
- [extra-05-tuning-clasic.ipynb](extra-05-tuning-clasic.ipynb): căutarea hiperparametrilor cu `GridSearchCV` (introducere în modulul 6).
- [extra-04-regresori-ml.ipynb](../04-regresie/extra-04-regresori-ml.ipynb): aceleași modele pentru regresie (prețuri).

---
## Puntea curs → seminar

**Concepte cheie** (modulul 5)

1. **Probabilitate și prag:** clasificatorul dă o probabilitate; pragul o transformă în clasă, iar 0,5 nu e o lege.
2. **Referința și împărțirea stratificată:** acuratețea se citește față de clasa majoritară.
3. **Matricea de confuzie:** fals pozitivi și fals negativi au costuri diferite; recall, precizie; AUC compară modele fără prag.
4. **Cum decid modelele:** regresia logistică (sigmoidă), kNN (vecini; cere scalare), arborele (reguli; memorează fără limită de adâncime), random forest și gradient boosting (mulți arbori), SVM (marginea cea mai largă; cere scalare).
5. **Arena:** același protocol (împărțire, `Pipeline`, referință, metrici) pentru toate modelele, într-un singur tabel; supraînvățarea se vede ca diferență mare între antrenare și test.

**Ce ceri asistentului AI** (exemplu de cerere)

> Pe `lab-set1`, fără `customerID`, cu `TotalCharges` transformat în număr (golurile = 0), împarte 80/20 stratificat (`random_state=42`). Într-un `Pipeline` cu completarea lipsurilor, one-hot și (unde trebuie) standardizare, compară clasa majoritară, regresia logistică, un arbore cu `max_depth=5` și random forest. Pentru fiecare: acuratețea pe antrenare și pe test, recall, precizie și AUC pe test, într-un singur tabel; apoi matricea de confuzie pentru modelul cel mai bun.

**Ce verifici înainte să accepți rezultatul**

- [ ] ID-urile sunt scoase, iar coloanele „numerice” stocate ca text au fost convertite;
- [ ] împărțirea este stratificată, iar pregătirea e în `Pipeline`, învățată doar pe antrenare;
- [ ] tabelul are un rând „clasa majoritară”;
- [ ] te uiți la recall, precizie și AUC, nu doar la acuratețe;
- [ ] compari antrenarea cu testul (un arbore nelimitat?);
- [ ] poți explica, pentru modelul ales, cum decide și ce greșeli face.

Continuăm cu [seminar-05-clasificare.ipynb](seminar-05-clasificare.ipynb).

## Jurnal de sesiune

Completează la sfârșit (sau cere agentului: *„închide capitolul”*) și păstrează-l în copia ta a notebook-ului. Dacă agentul rămâne fără tokeni, lipește primerul și acest jurnal unui alt agent și scrie „continuă de aici”.

- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):